# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs via the Croissant metadata.

In [ ]:
# List all available record sets and their details
if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets:
    print("Record Sets Available:")
    for rs in dataset.metadata.record_sets:
        print(f"@id: {rs.id}, Name: {getattr(rs, 'name', '')}, Description: {getattr(rs, 'description', '')}")
        print("  Fields:")
        for field in getattr(rs, 'fields', []):
            print(f"    - @id: {field.id}, Name: {getattr(field, 'name', '')}, DataType: {getattr(field, 'data_type', '')}")
        print("  Columns:")
        for column in getattr(rs, 'columns', []):
            print(f"    - @id: {column.id}, Name: {getattr(column, 'name', '')}, Source: {getattr(column, 'source', '')}")
else:
    print("No record sets found in the dataset. Inspecting distributions instead...")
    # Sometimes, in Croissant, the main data may be stored under distributions (files)
    if hasattr(dataset.metadata, 'distributions'):
        for dist in dataset.metadata.distributions:
            print(f"Distribution @id: {dist.id}, Name: {getattr(dist, 'name', '')}, Encoding: {getattr(dist, 'encoding_format', '')}, Content URL: {getattr(dist, 'content_url', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. If no record sets are available, attempt to load data from the main distribution.

In [ ]:
# Try to extract data from a record set if available, otherwise, handle a direct file
import warnings
record_set_ids = []
dataframes = {}

if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets:
    for rs in dataset.metadata.record_sets:
        record_set_ids.append(rs.id)
    # Use the first record set as an example
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record_set {record_set_id} with shape {df.shape}")
        except Exception as e:
            warnings.warn(f"Could not load records for record_set {record_set_id}: {e}")
    if record_set_ids:
        example_id = record_set_ids[0]
        print(f"\nColumns for record_set {example_id}:")
        print(dataframes[example_id].columns.tolist())
        display(dataframes[example_id].head())
else:
    # Fallback: try to load one of the distributions
    if hasattr(dataset.metadata, 'distributions'):
        from io import BytesIO
        import requests

        for dist in dataset.metadata.distributions:
            if getattr(dist, 'content_url', '').endswith('.csv'):
                url = getattr(dist, 'content_url', '')
                try:
                    resp = requests.get(url)
                    df = pd.read_csv(BytesIO(resp.content))
                    dataframes[dist.id] = df
                    print(f"Loaded DataFrame for distribution {dist.id} with shape {df.shape}")
                    print(f"Columns: {df.columns.tolist()}")
                    display(df.head())
                except Exception as e:
                    warnings.warn(f"Could not load CSV for distribution {dist.id}: {e}")
                break
    else:
        print("No record sets or distributions found with tabular data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.


In [ ]:
# Example EDA: Numeric filtering, normalization, grouping

import numpy as np

if dataframes:
    # Pick the first loaded DataFrame
    first_key = list(dataframes.keys())[0]
    df = dataframes[first_key]
    # Find numeric columns (float or int)
    numeric_col = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_col = col
            break
    if numeric_col is not None:
        print(f"Using numeric field '@id' (column): {numeric_col}\n")
        threshold = df[numeric_col].mean() if not np.isnan(df[numeric_col].mean()) else 0
        filtered_df = df[df[numeric_col] > threshold]
        print(f"Filtered records where {numeric_col} > {threshold:.2f} (mean): {len(filtered_df)} rows\n")
        # Normalization
        filtered_df[numeric_col + '_normalized'] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
        print(filtered_df[[numeric_col, numeric_col + '_normalized']].head())
        # Try grouping by a string field
        group_col = None
        for col in df.columns:
            if df[col].dtype == 'object' and col != numeric_col:
                group_col = col
                break
        if group_col:
            grouped = filtered_df.groupby(group_col)[numeric_col].mean().reset_index()
            print(f"\nMean of {numeric_col} by group '@id': {group_col}")
            print(grouped.head())
    else:
        print("No numeric columns found for EDA.")
else:
    print("No DataFrame loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_col is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_col].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_col}")
    plt.xlabel(numeric_col)
    plt.ylabel('Frequency')
    plt.show()

    if group_col:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_col, y=numeric_col)
        plt.title(f"{numeric_col} by {group_col}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. We reviewed the schema, record sets, and fields, and demonstrated basic exploratory data analysis and visualization using record set and field `@id` references where available.

For further data wrangling or advanced modeling, refer directly to schema documentation and ensure all references to entities use their stable `@id` as per the Croissant standard.